# 위험중립형 전략 백테스터 실행 (apps/backtester)

> 이 노트북은 `apps/backtester`의 pipeline/report 모듈을 그대로 호출해 백테스트를 실행하고 결과를 확인한다.
> 백테스트 로직 자체(`core/backtest`, `core/analytics`, `data/loaders`)는 손대지 않고, CLI 오케스트레이션 계층(`apps/backtester`)만 사용한다.
> 단계별 상세 분석/시각화가 필요하면 `notebooks/위험중립형_전략_백테스팅.ipynb`를 참고.

| 항목 | 설명 |
|---|---|
| 실행 모듈 | `apps.backtester.pipeline.run_backtest_pipeline` |
| 결과 저장 | `apps.backtester.report.save_report` → `report.md` / `metrics.json` / `figures/*.png` |
| 전략 파라미터 | PostgreSQL `strategies.params` (`risk_neutral`) |
| CLI 동급 명령 | `python -m apps.backtester run` ([README](../apps/backtester/README.md)) |


---

In [ ]:
import sys, os, warnings
from pathlib import Path

# 프로젝트 루트를 import 경로에 추가 (apps, core, data, storage 패키지 접근)
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'core').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')
print(f'✅ 프로젝트 루트: {PROJECT_ROOT}')


---
## 1. 백테스트 파라미터 설정

`apps.backtester.config.BacktesterConfig`가 CLI(`--start`, `--capital`, `--seed` 등)와 동일한 파라미터를 받는다.


In [ ]:
from datetime import date

from apps.backtester.config import BacktesterConfig

cfg = BacktesterConfig(
    strategy_name="risk_neutral",
    start_date=date(2018, 1, 1),
    end_date=date(2025, 12, 31),
    initial_capital=10_000_000.0,
    risk_free_rate=0.030,
    universe_size=5,
    rotation_size=2,
    rotation_interval_years=2,
    random_seed=42,
    output_dir=PROJECT_ROOT / "reports" / "backtester" / "notebook_run",
    save_charts=True,
)

print(f'전략          : {cfg.strategy_name}')
print(f'기간          : {cfg.start_date} ~ {cfg.end_date}')
print(f'초기 투자금    : {cfg.initial_capital:,.0f} 원')
print(f'랜덤 시드      : {cfg.random_seed}')
print(f'결과 저장 경로  : {cfg.output_dir}')


---
## 2. DB 연결 후 백테스트 파이프라인 실행

`apps/backtester/.env`에 DB 접속 정보(`POSTGRES_HOST`/`POSTGRES_USER`/`POSTGRES_PASSWORD`/`POSTGRES_DB`)가 설정되어 있어야 한다.
`strategies` 테이블에 `risk_neutral` 전략 파라미터가 미리 seed되어 있어야 한다 (`storage/postgres/schema/03_trader_schema.sql` → `06_strategy_seed.sql`).


In [ ]:
from apps.backtester.config import build_db_config, load_env
from apps.backtester.pipeline import run_backtest_pipeline
from storage.postgres.connection import PostgreDB

load_env(str(PROJECT_ROOT / "apps" / "backtester" / ".env"))

db = PostgreDB(build_db_config())
pipeline_result = run_backtest_pipeline(cfg, db)
db.close()


---
## 3. 결과 저장 (`report.md` / `metrics.json` / `figures/*.png`)


In [ ]:
from apps.backtester.report import save_report

output_dir = save_report(pipeline_result, cfg.output_dir, save_charts=cfg.save_charts)
print('저장된 파일:', sorted(p.name for p in output_dir.iterdir()))


---
## 4. 노트북에서 바로 확인

`save_report()`가 저장한 차트는 디스크에 닫혀서 남기 때문에, 노트북에서 보려면 `core.analytics`를 직접 호출해 다시 그린다.


In [ ]:
from IPython.display import display

from core.analytics.formatting import (
    build_compare_assets_table,
    build_kpi_table,
    build_performance_summary_table,
)
from core.analytics.performance import check_kpi_targets

result = pipeline_result.result
perf = pipeline_result.performance
status = check_kpi_targets(perf, result.config.strategy)

print(f'기간: {perf.start_date.date()} ~ {perf.end_date.date()}')
display(build_performance_summary_table(perf))
display(build_kpi_table(perf, status))

print()
print('비교 자산 성과 (위험중립형 vs 단기채 100% vs KOSPI)')
display(build_compare_assets_table(pipeline_result.compare_summary))


In [ ]:
import matplotlib.pyplot as plt

from core.analytics.visualization import plot_drawdown, plot_equity_curve, plot_monthly_returns

plot_equity_curve(result)
plt.show()

plot_drawdown(result)
plt.show()

plot_monthly_returns(result)
plt.show()


---
## 5. 저장된 마크다운 리포트 미리보기

`report.md`에는 KPI 표, Top Drawdown, Walk-Forward 윈도우, 유니버스 스냅샷, 투자자 해석 코멘터리가 모두 포함되어 있다.


In [ ]:
from IPython.display import Markdown, display

display(Markdown((output_dir / "report.md").read_text(encoding="utf-8")))
